# alignment

> Alignment service for temporal coordination via Silero VAD plugin

In [ ]:
#| default_exp alignment.services.alignment

In [ ]:
#| export
from typing import List, Dict, Any, Optional
import asyncio

from cjm_plugin_system.core.manager import PluginManager

from cjm_fasthtml_workflow_transcript_decomp.alignment.models import VADChunk
from cjm_fasthtml_workflow_transcript_decomp.decomposition.models import TextSegment

## AlignmentService

This service wraps the Silero VAD plugin to provide voice activity detection data. It processes audio files to extract time ranges where speech is detected, which are then used to align text segments with their corresponding audio positions.

In [ ]:
#| export
class AlignmentService:
    """Service for temporal alignment via Silero VAD plugin."""
    
    def __init__(
        self,
        plugin_manager: PluginManager,  # Plugin manager for accessing VAD plugin
        plugin_name: str = "cjm-media-plugin-silero-vad"  # Name of the VAD plugin
    ):
        """Initialize the alignment service."""
        self._manager = plugin_manager
        self._plugin_name = plugin_name
    
    def is_available(self) -> bool:  # True if plugin is loaded and ready
        """Check if the VAD plugin is available."""
        return self._manager.get_plugin(self._plugin_name) is not None
    
    def ensure_loaded(
        self,
        config: Optional[Dict[str, Any]] = None  # Optional plugin configuration
    ) -> bool:  # True if successfully loaded
        """Ensure the VAD plugin is loaded."""
        if self.is_available():
            return True
        
        # Try to find and load the plugin
        meta = self._manager.get_discovered_meta(self._plugin_name)
        if meta:
            return self._manager.load_plugin(meta, config or {"threshold": 0.5})
        return False
    
    async def analyze_audio_async(
        self,
        media_path: str  # Path to audio/video file
    ) -> tuple[List[VADChunk], float]:  # (VAD chunks, total duration)
        """Analyze audio file and return VAD chunks."""
        if not self.is_available():
            raise RuntimeError(f"Plugin {self._plugin_name} not loaded")
        
        # Execute plugin
        result = await self._manager.execute_plugin_async(
            self._plugin_name,
            media_path=media_path
        )
        
        # Extract ranges and metadata
        ranges = result.get('ranges', [])
        metadata = result.get('metadata', {})
        duration = metadata.get('duration', 0.0)
        
        # Convert to VADChunk objects
        chunks = []
        for idx, r in enumerate(ranges):
            chunk = VADChunk(
                index=idx,
                start_time=r.get('start', r.get('start_time', 0.0)),
                end_time=r.get('end', r.get('end_time', 0.0))
            )
            chunks.append(chunk)
        
        return chunks, duration
    
    def analyze_audio(
        self,
        media_path: str  # Path to audio/video file
    ) -> tuple[List[VADChunk], float]:  # (VAD chunks, total duration)
        """Analyze audio file synchronously."""
        return asyncio.get_event_loop().run_until_complete(
            self.analyze_audio_async(media_path)
        )

## Alignment Helpers

In [ ]:
#| export
def get_segments_without_time(
    segments:List[TextSegment],  # All segments
) -> List[TextSegment]:  # Segments missing time alignment
    """Get all segments that don't have time alignment."""
    return [s for s in segments if s.start_time is None or s.end_time is None]


#| export
def check_alignment_ready(
    segments:List[TextSegment],  # Text segments
    chunks:List[VADChunk],  # VAD chunks
) -> bool:  # True if counts match and alignment can proceed
    """Check if segment and VAD chunk counts match for 1:1 alignment."""
    return len(segments) == len(chunks)


#| export
def populate_segment_times(
    segments:List[TextSegment],  # Text segments to populate (modified in place)
    chunks:List[VADChunk],  # VAD chunks providing timestamps
) -> bool:  # True if times were populated (counts matched)
    """Copy timestamps from VAD chunks to segments using 1:1 index mapping."""
    if not check_alignment_ready(segments, chunks):
        return False
    
    for segment, chunk in zip(segments, chunks):
        segment.start_time = chunk.start_time
        segment.end_time = chunk.end_time
    
    return True

## Tests

The following cells demonstrate the alignment service and helper functions.

In [ ]:
# Test get_segments_without_time
segments = [
    TextSegment(index=0, text="Has time", start_time=0.0, end_time=1.0),
    TextSegment(index=1, text="No time"),  # Missing time alignment
    TextSegment(index=2, text="Also has time", start_time=2.0, end_time=3.0)
]

no_time = get_segments_without_time(segments)
print(f"Segments without time: {[s.index for s in no_time]}")

In [ ]:
# Test check_alignment_ready
segments_3 = [TextSegment(index=i, text=f"Seg {i}") for i in range(3)]
chunks_3 = [VADChunk(index=i, start_time=i*1.0, end_time=(i+1)*1.0) for i in range(3)]
chunks_5 = [VADChunk(index=i, start_time=i*1.0, end_time=(i+1)*1.0) for i in range(5)]

print(f"3 segments vs 3 chunks: {check_alignment_ready(segments_3, chunks_3)}")  # True
print(f"3 segments vs 5 chunks: {check_alignment_ready(segments_3, chunks_5)}")  # False

In [ ]:
# Test populate_segment_times - matching counts
segments = [TextSegment(index=i, text=f"Segment {i}") for i in range(3)]
chunks = [
    VADChunk(index=0, start_time=0.0, end_time=1.5),
    VADChunk(index=1, start_time=1.5, end_time=3.0),
    VADChunk(index=2, start_time=3.0, end_time=5.0),
]

print("Before populate_segment_times:")
for s in segments:
    print(f"  Segment {s.index}: {s.start_time} - {s.end_time}")

result = populate_segment_times(segments, chunks)
print(f"\nPopulate result: {result}")

print("\nAfter populate_segment_times:")
for s in segments:
    print(f"  Segment {s.index}: {s.start_time} - {s.end_time}")

In [ ]:
# Test populate_segment_times - mismatched counts (should fail)
segments_2 = [TextSegment(index=i, text=f"Segment {i}") for i in range(2)]
chunks_4 = [VADChunk(index=i, start_time=i*1.0, end_time=(i+1)*1.0) for i in range(4)]

result = populate_segment_times(segments_2, chunks_4)
print(f"Populate with mismatched counts: {result}")  # False
print(f"Segment 0 times unchanged: {segments_2[0].start_time} - {segments_2[0].end_time}")  # None - None

### AlignmentService with Silero VAD Plugin

These tests require the Silero VAD plugin to be installed and discoverable.

In [ ]:
#| eval: false
# Test AlignmentService with Silero VAD plugin
from pathlib import Path
from cjm_plugin_system.core.manager import PluginManager

# Calculate project root from notebook location (nbs/services/ -> project root)
project_root = Path.cwd().parent.parent
manifests_dir = project_root / ".cjm" / "manifests"

# Create plugin manager with explicit search path
manager = PluginManager(search_paths=[manifests_dir])
manager.discover_manifests()

print(f"Discovered {len(manager.discovered)} plugins from {manifests_dir}")

# Check if VAD plugin is available
vad_meta = manager.get_discovered_meta("cjm-media-plugin-silero-vad")
if vad_meta:
    print(f"Found plugin: {vad_meta.name} v{vad_meta.version}")
else:
    print("Silero VAD plugin not found - install via plugins.yaml")

[PluginManager] Discovered manifest: cjm-transcription-plugin-voxtral-hf from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests/cjm-transcription-plugin-voxtral-hf.json
[PluginManager] Discovered manifest: cjm-system-monitor-nvidia from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests/cjm-system-monitor-nvidia.json
[PluginManager] Discovered manifest: cjm-transcription-plugin-whisper from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests/cjm-transcription-plugin-whisper.json
[PluginManager] Discovered manifest: cjm-media-plugin-silero-vad from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests/cjm-media-plugin-silero-vad.json
[PluginManager] Discovered manifest: cjm-graph-plugin-sqlite from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manif

Discovered 6 plugins from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-fasthtml-workflow-transcript-decomp/.cjm/manifests
Found plugin: cjm-media-plugin-silero-vad v0.0.2


In [ ]:
#| eval: false
# Initialize and test AlignmentService
if vad_meta:
    # Load the plugin
    manager.load_plugin(vad_meta, {"threshold": 0.5})
    
    align_service = AlignmentService(manager)
    print(f"Plugin available: {align_service.is_available()}")
    
    # Test audio analysis (use await directly - Jupyter supports top-level await)
    audio_file = "/mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcription-plugin-voxtral-hf/test_files/02 - 1. Laying Plans.mp3"
    
    chunks, duration = await align_service.analyze_audio_async(audio_file)
    
    print(f"\nAudio duration: {duration:.2f}s")
    print(f"Detected {len(chunks)} VAD chunks")
    print("\nFirst 5 chunks:")
    for chunk in chunks[:5]:
        print(f"  [{chunk.index}] {chunk.start_time:.2f}s - {chunk.end_time:.2f}s (duration: {chunk.duration:.2f}s)")

[PluginManager] Launching worker for cjm-media-plugin-silero-vad...


[cjm-media-plugin-silero-vad] Starting worker on port 55131...
[cjm-media-plugin-silero-vad] Logs: /home/innom-dt/.cjm/logs/cjm-media-plugin-silero-vad.log


[PluginManager] HTTP Request: GET http://127.0.0.1:55131/health "HTTP/1.1 200 OK"
[PluginManager] HTTP Request: POST http://127.0.0.1:55131/initialize "HTTP/1.1 200 OK"
[PluginManager] Loaded plugin: cjm-media-plugin-silero-vad
[PluginManager] HTTP Request: POST http://127.0.0.1:55131/execute "HTTP/1.1 200 OK"


[cjm-media-plugin-silero-vad] Worker ready.
Plugin available: True

Audio duration: 256.39s
Detected 89 VAD chunks

First 5 chunks:
  [0] 1.10s - 2.30s (duration: 1.20s)
  [1] 4.80s - 5.90s (duration: 1.10s)
  [2] 6.60s - 9.80s (duration: 3.20s)
  [3] 10.70s - 12.40s (duration: 1.70s)
  [4] 12.60s - 15.40s (duration: 2.80s)


In [ ]:
#| eval: false
# Cleanup
if vad_meta:
    manager.unload_all()
    print("Plugins unloaded")

[PluginManager] HTTP Request: POST http://127.0.0.1:55131/cleanup "HTTP/1.1 200 OK"
[PluginManager] Unloaded plugin: cjm-media-plugin-silero-vad


Plugins unloaded


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()